# Parallel processing with Pastastore

This notebook shows parallel processing capabilities of `PastaStore`.

Parallel processing is platform dependent and may not
always work. On Python 3.14 and higher, `ProcessPoolExecutor` no longer defaults
to `fork`, so worker processes started from a notebook cannot import functions
defined only in notebook cells. For parallel `.apply()` examples, use importable
functions from a regular Python module instead of notebook-local callables.

Built-in methods such as `solve_models()` and `get_statistics()` continue to work
in notebooks. For custom functions, move the worker function into a `.py` file or
run the code from a Python script.

For Windows users, parallel solving does not work when called directly from
Jupyter Notebooks or IPython. To use parallel solving on Windows, the following
code should be used in a Python file.

```python
from multiprocessing import freeze_support

if __name__ == "__main__":
    freeze_support()
    pstore.apply("models", some_func, parallel=True)
```

In [1]:
import pastas as ps

import pastastore as pst
from pastastore import parallel
from pastastore.datasets import example_pastastore

ps.set_log_level("ERROR")  # silence Pastas logger for this notebook
pst.get_color_logger("INFO", "pastastore")
pst.show_versions()

Pastastore version : 2.0.0

Python version     : 3.14.2
Pandas version     : 2.3.3
Matplotlib version : 3.10.8
Pastas version     : 2.0.0rc1
PyYAML version     : 6.0.3



## Example pastastore

Load some example data, create models and solve them to showcase parallel processing.

In [2]:
# get the example pastastore
conn = pst.PasConnector("my_connector", "./temp")
# conn = pst.ArcticDBConnector("my_connector", "lmdb://./temp")
pstore = example_pastastore(conn)
parallel_worker_kwargs = (
    {"connector": pstore.conn} if pstore.conn.conn_type == "pas" else {}
)
pstore.create_models_bulk();

PasConnector: library 'oseries' created in '/home/david/github/pastastore/docs/notebooks/temp/my_connector/oseries'
PasConnector: library 'stresses' created in '/home/david/github/pastastore/docs/notebooks/temp/my_connector/stresses'
PasConnector: library 'models' created in '/home/david/github/pastastore/docs/notebooks/temp/my_connector/models'
PasConnector: library 'oseries_models' created in '/home/david/github/pastastore/docs/notebooks/temp/my_connector/oseries_models'
PasConnector: library 'stresses_models' created in '/home/david/github/pastastore/docs/notebooks/temp/my_connector/stresses_models'


Bulk creation models:   0%|          | 0/5 [00:00<?, ?it/s]

Stressmodel 'recharge' added to model 'head_nb5'.
Stressmodel 'recharge' added to model 'oseries2'.
Stressmodel 'recharge' added to model 'head_mw'.
Stressmodel 'recharge' added to model 'oseries1'.
Stressmodel 'recharge' added to model 'oseries3'.


## Solving models

The `PastaStore.solve_models()` method supports parallel processing.

In [3]:
pstore.solve_models(parallel=True)

Solving models (parallel):   0%|          | 0/5 [00:00<?, ?it/s]

Parameter 'recharge_f' on upper bound: 0.00e+00
The series 'oseries1' has nan-values. Pastas will use the `fill_nan` from the StressModel's settings (ps.timeseries.settings) parsed to the TimeSeries settings to fill up the nan-values.
Response tmax for 'recharge' > than warmup period.


## Parallel processing using `.apply()`

We define a function that takes a name as input and returns some result. In this case,
return the $R^2$ value for each model.

In [4]:
def rsq(model_name: str) -> float:
    """Compute the R-squared value of a Pastas model."""
    ml = pstore.get_models(model_name)
    return ml.stats.rsq()

We can apply this function to all models in the pastastore using `pstore.apply()`. 
By default this function is run sequentially. 

In [5]:
pstore.apply("models", rsq, progressbar=True)

Computing rsq:   0%|          | 0/5 [00:00<?, ?it/s]

head_nb5    0.438024
oseries2    0.931883
head_mw     0.159337
oseries1    0.904481
oseries3    0.030468
dtype: float64

In order to run a custom function in parallel, the worker must be importable by the
child process. The notebook-defined `rsq()` function above works sequentially, but
for parallel execution on Python 3.14+ we call the importable helper from
`pastastore.parallel`.

In [6]:
parallel.rsq??

Signature: parallel.rsq(model_name: 'str', connector: 'BaseConnector | None' = None) -> 'float'
Source:   
def rsq(model_name: str, connector: BaseConnector | None = None) -> float:
    """Compute the R-squared value of a Pastas model."""
    return model_statistic(model_name, "rsq", connector)
File:      ~/github/pastastore/pastastore/parallel.py
Type:      function

In [7]:
pstore.apply(
    "models",
    parallel.rsq,
    kwargs=parallel_worker_kwargs,
    progressbar=True,
    parallel=True,
)

Computing rsq (parallel):   0%|          | 0/5 [00:00<?, ?it/s]

The series 'oseries1' has nan-values. Pastas will use the `fill_nan` from the StressModel's settings (ps.timeseries.settings) parsed to the TimeSeries settings to fill up the nan-values.


head_nb5    0.438024
oseries2    0.931883
head_mw     0.159337
oseries1    0.904481
oseries3    0.030468
dtype: float64

## Get model statistics

The function `pstore.get_statistics` also supports parallel processing.

In [8]:
pstore.get_statistics(["rsq", "mae"])

,rsq,mae
head_nb5,0.438024,0.318404
oseries2,0.931883,0.087067
head_mw,0.159337,0.632279
oseries1,0.904481,0.091288
oseries3,0.030468,0.106254


In [9]:
pstore.get_statistics(["rsq", "mae"], parallel=True)

The series 'oseries1' has nan-values. Pastas will use the `fill_nan` from the StressModel's settings (ps.timeseries.settings) parsed to the TimeSeries settings to fill up the nan-values.


,rsq,mae
_get_statistics,,
head_nb5,0.438024,0.318404
oseries2,0.931883,0.087067
head_mw,0.159337,0.632279
oseries1,0.904481,0.091288
oseries3,0.030468,0.106254


## Compute prediction intervals

Let's try using a more complex function and passing that to apply to use
parallel processing. In this case we want to compute the prediction interval,
and pass along the $\alpha$ value via the keyword arguments.

In [10]:
def prediction_interval(model_name, **kwargs):
    """Compute the prediction interval for a Pastas model."""
    ml = pstore.get_models(model_name)
    return ml.solver.prediction_interval(**kwargs)

In [11]:
pstore.apply("models", prediction_interval, kwargs={"alpha": 0.05})

Computing prediction_interval:   0%|          | 0/5 [00:00<?, ?it/s]

head_nb5           oseries2         head_mw           oseries1  \
               0.025     0.975    0.025 0.975     0.025     0.975    0.025   
1960-04-29       NaN       NaN      NaN   NaN  6.316634  9.657977      NaN   
1960-04-30       NaN       NaN      NaN   NaN  6.190078  9.447394      NaN   
1960-05-01       NaN       NaN      NaN   NaN  6.221171  9.444635      NaN   
1960-05-02       NaN       NaN      NaN   NaN  6.344872  9.521428      NaN   
1960-05-03       NaN       NaN      NaN   NaN  6.202043  9.444549      NaN   
...              ...       ...      ...   ...       ...       ...      ...   
2015-06-29  7.466300  9.095025      NaN   NaN  6.431250  9.633844      NaN   
2015-12-27  7.968414  9.686361      NaN   NaN       NaN       NaN      NaN   
2016-07-01  7.693290  9.438314      NaN   NaN       NaN       NaN      NaN   
2019-03-28  8.036616  9.709772      NaN   NaN       NaN       NaN      NaN   
2021-12-22       NaN       NaN      NaN   NaN       NaN       NaN      NaN   

                 oseries3        
           0.975    0.025 0.975  
1960-04-29   NaN      NaN   NaN  
1960-04-30   NaN      NaN   NaN  
1960-05-01   NaN      NaN   NaN  
1960-05-02   NaN      NaN   NaN  
1960-05-03   NaN      NaN   NaN  
...          ...      ...   ...  
2015-06-29   NaN      NaN   NaN  
2015-12-27   NaN      NaN   NaN  
2016-07-01   NaN      NaN   NaN  
2019-03-28   NaN      NaN   NaN  
2021-12-22   NaN      NaN   NaN  

[20154 rows x 10 columns]

In [12]:
pstore.apply(
    "models",
    parallel.prediction_interval,
    kwargs={**parallel_worker_kwargs, "alpha": 0.05},
    parallel=True,
)

Computing prediction_interval (parallel):   0%|          | 0/5 [00:00<?, ?it/s]

The series 'oseries1' has nan-values. Pastas will use the `fill_nan` from the StressModel's settings (ps.timeseries.settings) parsed to the TimeSeries settings to fill up the nan-values.


head_nb5           oseries2         head_mw           oseries1  \
               0.025     0.975    0.025 0.975     0.025     0.975    0.025   
1960-04-29       NaN       NaN      NaN   NaN  6.280527  9.502628      NaN   
1960-04-30       NaN       NaN      NaN   NaN  6.236858  9.616544      NaN   
1960-05-01       NaN       NaN      NaN   NaN  6.345629  9.508508      NaN   
1960-05-02       NaN       NaN      NaN   NaN  6.193932  9.510928      NaN   
1960-05-03       NaN       NaN      NaN   NaN  6.253329  9.537718      NaN   
...              ...       ...      ...   ...       ...       ...      ...   
2015-06-29  7.343683  9.128056      NaN   NaN  6.533454  9.568842      NaN   
2015-12-27  7.950715  9.623954      NaN   NaN       NaN       NaN      NaN   
2016-07-01  7.729652  9.501949      NaN   NaN       NaN       NaN      NaN   
2019-03-28  7.926338  9.748687      NaN   NaN       NaN       NaN      NaN   
2021-12-22       NaN       NaN      NaN   NaN       NaN       NaN      NaN   

                 oseries3        
           0.975    0.025 0.975  
1960-04-29   NaN      NaN   NaN  
1960-04-30   NaN      NaN   NaN  
1960-05-01   NaN      NaN   NaN  
1960-05-02   NaN      NaN   NaN  
1960-05-03   NaN      NaN   NaN  
...          ...      ...   ...  
2015-06-29   NaN      NaN   NaN  
2015-12-27   NaN      NaN   NaN  
2016-07-01   NaN      NaN   NaN  
2019-03-28   NaN      NaN   NaN  
2021-12-22   NaN      NaN   NaN  

[20154 rows x 10 columns]

## Get signatures

The function `pstore.get_signatures` does not explicitly support parallel processing but can be used in combination with `pstore.apply`

In [13]:
signatures = [
    "cv_period_mean",
    "cv_date_min",
    "cv_date_max",
    "cv_fall_rate",
    "cv_rise_rate",
]

In [14]:
pstore.get_signatures(signatures=signatures)

,head_nb5,oseries2,head_mw,oseries1,oseries3
cv_period_mean,0.061879,0.015199,0.145062,0.013066,0.029168
cv_date_min,0.246021,0.128636,0.254627,0.145884,1.394852
cv_date_max,1.262425,0.722945,1.083929,0.300328,0.444442
cv_fall_rate,-1.136450,-0.722718,-1.430200,-0.744797,-1.032837
cv_rise_rate,1.259450,0.836678,1.097257,0.862981,0.931181


In [15]:
pstore.apply(
    "oseries", pstore.get_signatures, kwargs={"signatures": signatures}, parallel=True
)

Computing get_signatures (parallel):   0%|          | 0/5 [00:00<?, ?it/s]

get_signatures,head_nb5,oseries2,head_mw,oseries1,oseries3
cv_period_mean,0.061879,0.015199,0.145062,0.013066,0.029168
cv_date_min,0.246021,0.128636,0.254627,0.145884,1.394852
cv_date_max,1.262425,0.722945,1.083929,0.300328,0.444442
cv_fall_rate,-1.136450,-0.722718,-1.430200,-0.744797,-1.032837
cv_rise_rate,1.259450,0.836678,1.097257,0.862981,0.931181


## Load models

Load models in parallel.

In [16]:
pstore.apply("models", pstore.get_models)

Computing get_models:   0%|          | 0/5 [00:00<?, ?it/s]

{'head_nb5': Model(oseries=head_nb5, name=head_nb5, constant=True, noisemodel=False),
 'oseries2': Model(oseries=oseries2, name=oseries2, constant=True, noisemodel=False),
 'head_mw': Model(oseries=head_mw, name=head_mw, constant=True, noisemodel=False),
 'oseries1': Model(oseries=oseries1, name=oseries1, constant=True, noisemodel=False),
 'oseries3': Model(oseries=oseries3, name=oseries3, constant=True, noisemodel=False)}

The `max_workers` keyword argument sets the number of workers that are spawned. The default value is often fine, but it can be set explicitly.

The following works for `PasConnector`. See alternative code below for `ArcticDBConnector`.  

In [17]:
pstore.apply("models", pstore.get_models, parallel=True, max_workers=5)

Computing get_models (parallel):   0%|          | 0/5 [00:00<?, ?it/s]

The series 'oseries1' has nan-values. Pastas will use the `fill_nan` from the StressModel's settings (ps.timeseries.settings) parsed to the TimeSeries settings to fill up the nan-values.


{'head_nb5': Model(oseries=head_nb5, name=head_nb5, constant=True, noisemodel=False),
 'oseries2': Model(oseries=oseries2, name=oseries2, constant=True, noisemodel=False),
 'head_mw': Model(oseries=head_mw, name=head_mw, constant=True, noisemodel=False),
 'oseries1': Model(oseries=oseries1, name=oseries1, constant=True, noisemodel=False),
 'oseries3': Model(oseries=oseries3, name=oseries3, constant=True, noisemodel=False)}

## Storing models in parallel
<div class="alert alert-info">
<strong>Note</strong>

This section is mostly for the developer so he doesn't forget why and how 
delayed updating of the model links was implemented.
</div>

We want to build and solve our time series models in 2 steps, first without a noise
model and then with a noise model, and then store the result. We empty the models
library to start from scratch.

In [18]:
pstore.empty_library("models", prompt=False, progressbar=False)

Emptied library models in my_connector: <class 'pastastore.connectors.PasConnector'>
Emptied library oseries_models in my_connector: <class 'pastastore.connectors.PasConnector'>
Emptied library stresses_models in my_connector: <class 'pastastore.connectors.PasConnector'>


In the first example we apply the function in parallel using the importable
`parallel.two_step_solve` worker. A separate recomputation is
performed after the parallel apply to update the links between the time series
names and the models.

The `conn._added_models` keeps track of added models so that the model links can be
updated after all models have been added. In parallel mode, the child processes do not
have access to this variable in the main thread, meaning it is not updated.

In [19]:
pstore.conn._added_models

[]

In [20]:
# check if update flags were reset after adding models links after parallel apply
print(f"{pstore.conn._oseries_links_need_update.value = }")
print(f"{pstore.conn._stresses_links_need_update.value = }")

pstore.conn._oseries_links_need_update.value = False
pstore.conn._stresses_links_need_update.value = False


In [21]:
pstore.apply(
    "oseries",
    parallel.two_step_solve,
    kwargs=parallel_worker_kwargs,
    parallel=True,
    max_workers=2,
)

Computing two_step_solve (parallel):   0%|          | 0/5 [00:00<?, ?it/s]

Stressmodel 'recharge' added to model 'oseries2'.
Stressmodel 'recharge' added to model 'head_nb5'.
Stressmodel 'recharge' added to model 'head_mw'.
Stressmodel 'recharge' added to model 'oseries1'.


The series 'oseries1' has nan-values. Pastas will use the `fill_nan` from the StressModel's settings (ps.timeseries.settings) parsed to the TimeSeries settings to fill up the nan-values.
Response tmax for 'recharge' > than warmup period.


Stressmodel 'recharge' added to model 'oseries3'.


Response tmax for 'recharge' > than warmup period.
Parameter 'recharge_f' on upper bound: 0.00e+00
Parameter 'recharge_f' on upper bound: 0.00e+00


As expected, the list remains empty:

In [22]:
pstore.conn._added_models

[]

In [23]:
pstore.models

<ModelAccessor> 5 model(s): 
['head_nb5', 'oseries2', 'head_mw', 'oseries1', 'oseries3']

The parallel apply automatically updates the model links libraries, so the update flags
will be equal to False if running on Linux and Python 3.13 or lower. On Windows/MacOS
and/or Python 3.14+, this value will still be True.

In [24]:
# check if update flags were reset after adding models links after parallel apply
print(f"{pstore.conn._oseries_links_need_update.value = }")
print(f"{pstore.conn._stresses_links_need_update.value = }")

pstore.conn._oseries_links_need_update.value = True
pstore.conn._stresses_links_need_update.value = True


Let's check the `oseries_models` result:

In [25]:
pstore.oseries_models

{'head_nb5': ['head_nb5'],
 'oseries2': ['oseries2'],
 'head_mw': ['head_mw'],
 'oseries1': ['oseries1'],
 'oseries3': ['oseries3']}

In [26]:
# check if update flags were reset after adding models links after parallel apply
print(f"{pstore.conn._oseries_links_need_update.value = }")
print(f"{pstore.conn._stresses_links_need_update.value = }")

pstore.conn._oseries_links_need_update.value = False
pstore.conn._stresses_links_need_update.value = False


Now we repeat the process with `parallel=False`. Now the `_added_models` attribute can
be updated properly since there is only the main instance of PastaStore.

Once again, we empty the models library to start fresh.

In [27]:
def two_step_solve(name: str) -> None:
    """Solve a Pastas model in two steps and store the result."""
    ml = pstore.create_model(name)
    ml.solve(report=False)
    ml.add_noisemodel(ps.ArNoiseModel())
    ml.solve(initial=False, report=False)
    pstore.add_model(ml, overwrite=True)

In [28]:
pstore.empty_library("models", prompt=False, progressbar=False)

Emptied library models in my_connector: <class 'pastastore.connectors.PasConnector'>
Emptied library oseries_models in my_connector: <class 'pastastore.connectors.PasConnector'>
Emptied library stresses_models in my_connector: <class 'pastastore.connectors.PasConnector'>


In [29]:
pstore.apply("oseries", two_step_solve, parallel=False)

Computing two_step_solve:   0%|          | 0/5 [00:00<?, ?it/s]

Stressmodel 'recharge' added to model 'head_nb5'.
Stressmodel 'recharge' added to model 'oseries2'.
Stressmodel 'recharge' added to model 'head_mw'.
Stressmodel 'recharge' added to model 'oseries1'.
Stressmodel 'recharge' added to model 'oseries3'.


The `_added_models` attribute should now contain the names of all 5 models.

In [30]:
pstore.conn._added_models

['head_nb5', 'oseries2', 'head_mw', 'oseries1', 'oseries3']

The update flags should be set to True, which should trigger an update once we try to
access the out-of-date data.

In [31]:
# check if update flags were reset after adding models links after parallel apply
print(f"{pstore.conn._oseries_links_need_update.value = }")
print(f"{pstore.conn._stresses_links_need_update.value = }")

pstore.conn._oseries_links_need_update.value = True
pstore.conn._stresses_links_need_update.value = True


Now let's trigger the update by looking at `oseries_models`. This will update the
database, empty the `_added_models` attribute and set the update flags to False.

In [32]:
pstore.oseries_models

{'head_nb5': ['head_nb5'],
 'oseries2': ['oseries2'],
 'head_mw': ['head_mw'],
 'oseries1': ['oseries1'],
 'oseries3': ['oseries3']}

In [33]:
pstore.conn._added_models

[]

In [34]:
# check if update flags were reset after adding models links after parallel apply
print(f"{pstore.conn._oseries_links_need_update.value = }")
print(f"{pstore.conn._stresses_links_need_update.value = }")

pstore.conn._oseries_links_need_update.value = False
pstore.conn._stresses_links_need_update.value = False


## ArcticDBConnector workaround

For `ArcticDBConnector`, the underlying database connection objects cannot be pickled, which is required for Python's multiprocessing. Therefore, passing methods directly from the `PastaStore` or `ArcticDBConnector` classes will not work in parallel mode.

**The workaround:** The `_parallel()` method uses an initializer that creates a new connector instance in each worker process and stores it in the connectors module. Custom worker functions should be defined in an importable Python module and obtain the worker-local connector from there.

This is the standard Python pattern for using unpicklable objects with multiprocessing. See the [Python documentation](https://docs.python.org/3/library/concurrent.futures.html#processpoolexecutor) for more details.

**Note:** If you need access to methods from the `PastaStore` class, create a new one inside the worker from the worker-local connector.

**Example:** The notebook imports `parallel.get_model`, which constructs `PastaStore(connector)` inside the worker using either the passed `PasConnector` or the worker-local ArcticDB connector.

In [35]:
parallel.get_model??

Signature:
parallel.get_model(
    model_name: 'str',
    connector: 'BaseConnector | None' = None,
)
Source:   
def get_model(model_name: str, connector: BaseConnector | None = None):
    """Load a model from the active worker connector."""
    resolved_connector = _resolve_connector(connector)
    return resolved_connector.get_models(model_name)
File:      ~/github/pastastore/pastastore/parallel.py
Type:      function

In [36]:
pstore.apply(
    "models",
    parallel.get_model,
    kwargs=parallel_worker_kwargs,
    fancy_output=True,
    parallel=True,
    max_workers=5,
)

Computing get_model (parallel):   0%|          | 0/5 [00:00<?, ?it/s]

The series 'oseries1' has nan-values. Pastas will use the `fill_nan` from the StressModel's settings (ps.timeseries.settings) parsed to the TimeSeries settings to fill up the nan-values.


{'head_nb5': Model(oseries=head_nb5, name=head_nb5, constant=True, noisemodel=True),
 'oseries2': Model(oseries=oseries2, name=oseries2, constant=True, noisemodel=True),
 'head_mw': Model(oseries=head_mw, name=head_mw, constant=True, noisemodel=True),
 'oseries1': Model(oseries=oseries1, name=oseries1, constant=True, noisemodel=True),
 'oseries3': Model(oseries=oseries3, name=oseries3, constant=True, noisemodel=True)}

## Clean up

Clean up temporary pastastore.

In [37]:
pst.util.delete_pastastore(pstore)

Deleting PasConnector database: 'my_connector' ... 
Done!
